In [29]:
# Install Packages
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-core
!pip install -q langchain-ollama
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q pypdf
!pip install -q langchain-text-splitters

# Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM

# **Load PDF**

In [30]:
loader = PyPDFLoader("employee_policy.pdf")
documents = loader.load()

print("Total Pages:", len(documents))
print("\nFirst Page Preview:\n")
print(documents[0].page_content[:500])


Total Pages: 1

First Page Preview:

Leave Policy  
Employees receive 12 casual leaves annually. 
 Employees receive 15 sick leaves annually.  
Unused casual leaves cannot be carried forward.  
 
Work From Home Policy  
Employees may work from home twice per week.  
Manager approval is required for additional remote work.  
 
Travel Policy Travel expenses are reimbursed within 30 days.  
Original receipts must be submitted for reimbursement. 
 
 Medical Insurance Policy  
 
All employees are covered under company medical insurance.


# **Chunking**

In [38]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len
)

chunks = splitter.split_documents(documents)

print("\nTotal Chunks:", len(chunks))



Total Chunks: 8


# **Embeddings**

In [39]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# **Create FAISS VECTOR STORE**

In [40]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS Vector Store Created Successfully")

FAISS Vector Store Created Successfully


# **Create Retriver**

In [41]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever Ready")

Retriever Ready


# **Test Retrival**

In [42]:
query = "How many casual leaves are allowed?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):
    print(f"\nChunk {i+1}")
    print("-" * 80)
    print(doc.page_content)


Chunk 1
--------------------------------------------------------------------------------
Employees receive 15 sick leaves annually.  
Unused casual leaves cannot be carried forward.

Chunk 2
--------------------------------------------------------------------------------
Leave Policy  
Employees receive 12 casual leaves annually.

Chunk 3
--------------------------------------------------------------------------------
Work From Home Policy  
Employees may work from home twice per week.


# **Connect Ollama**

In [43]:
llm = OllamaLLM(model="llama3")
print("Connected to ollama")

Connected to ollama


# **Build Context**

In [45]:
# Build Context

context = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

print("Retrieved Context:")
print(context)

# Create Prompt

prompt = f"""
You are an HR Assistant.

Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

# Generate Response

response = llm.invoke(prompt)

print("\nFinal Answer:")
print(response)

Retrieved Context:
Employees receive 15 sick leaves annually.  
Unused casual leaves cannot be carried forward.

Leave Policy  
Employees receive 12 casual leaves annually.

Work From Home Policy  
Employees may work from home twice per week.


ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b1674076f90>: Failed to establish a new connection: [Errno 111] Connection refused'))